In [0]:
%sql
-- Clean up files older than 7 days (default)
VACUUM filestore.silver.employees_enriched;

-- Collect stats for the optimizer
ANALYZE TABLE filestore.silver.employees_enriched COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
# Create a Volume to store raw landing data securely
spark.sql("CREATE VOLUME IF NOT EXISTS filestore.bronze.landing_zone")

# Move a file from legacy DBFS to the new Volume
# Volumes use the path format: /Volumes/catalog/schema/volume_name/
dbutils.fs.cp("/FileStoreWasim/hr/raw/employees.csv", "/Volumes/filestore/bronze/landing_zone/employees.csv")

print("File migrated to Unity Catalog Volume.")

In [0]:
%sql
-- See all inserts, updates, and deletes between two versions
SELECT * FROM table_changes('filestore.silver.employees_enriched', 1, 2)
ORDER BY _commit_timestamp;

In [0]:
%sql
-- Create a complete sandbox copy for testing new logic
CREATE TABLE filestore.silver.employees_enriched_test 
DEEP CLONE filestore.silver.employees_enriched;

In [0]:
# Example of a Master Orchestrator Notebook
# This allows you to pass parameters between tasks
dbutils.notebook.run("Bronze_Ingestion_Notebook", 600, {"input_path": "/FileStoreWasim/hr/raw/"})
dbutils.notebook.run("Silver_Transformation_Notebook", 600)
dbutils.notebook.run("Gold_Reporting_Notebook", 600)